# DAEN 328 Assignment 5 Guide: Load and Serve

This guide teaches every skill Assignment 5 needs, using a **small practice dataset** (a coffee shop's orders). Nothing in this notebook is graded and you do not submit it.

**How to use it:** run each cell, read the explanation, and change things to see what happens. Every pattern you see here (loading lookup tables, avoiding duplicates on a re-run, turning text dates into real dates, rejecting bad rows, writing SQL, building a Streamlit dashboard) is exactly what you will then write yourself for `EmployeeDB` in your own Assignment 5 notebook and `app.py`.

The practice data is deliberately a **different** business with **different** tables. You cannot copy this code into your assignment unchanged; you have to understand it and adapt it. That is the point.

| Part | You learn | You then do (in your own files) |
|---|---|---|
| A. Load | How a Python loader moves a CSV into related tables, safely | Load your Assignment 3 `Cleaned_data.csv` into the 7 `EmployeeDB` tables |
| B. SQL | JOINs, GROUP BY, HAVING, subqueries, dates, window functions | Answer 10 business questions about KLB with your own SQL |
| C. Serve | How Streamlit re-runs, caching, filters, parameterized queries, Plotly charts | Build `app.py`, a dashboard on `EmployeeDB` |

**Before you start:** Assignment 4 is done, so `EmployeeDB` exists with your 7 empty tables. Put this notebook, `practice_orders.csv` and `practice_app.py` in one folder.

## Setup: libraries

Run this once inside your course environment (`daen328_env`). Streamlit 1.50 or newer is needed for the dashboard code in Part C.

In [1]:
# Uncomment and run once if any import below fails:
# !pip install --upgrade pandas psycopg2-binary sqlalchemy "streamlit>=1.50" plotly

In [2]:
import pandas as pd
import psycopg2
from sqlalchemy import create_engine, text

## CONFIG: edit this cell once

Every connection in this notebook uses these variables. Put your own PostgreSQL password here, and keep `DB_NAME = "EmployeeDB"`. The practice tables go into their own **schema** (think of a schema as a folder inside a database) called `practice`, so they never mix with your 7 assignment tables in the `public` schema.

In [3]:
DB_NAME = "EmployeeDB"
DB_USER = "postgres"
DB_PASSWORD = "admin"      # change to the password YOU set when installing PostgreSQL
DB_HOST = "localhost"
DB_PORT = "5432"

DB_PARAMS = {"dbname": DB_NAME, "user": DB_USER, "password": DB_PASSWORD,
             "host": DB_HOST, "port": DB_PORT}
engine = create_engine(f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}")

# Part A: Loading a CSV into related tables

## A1. The big picture

A CSV is one wide table. A normalized database is several narrow tables linked by keys. A **loader** is the program that splits each CSV row into the right pieces and puts each piece into the right table, in the right order.

The order matters because of foreign keys. A row in `orders` points at a store with `storeid`. That store must exist **before** the order that points at it. So a loader always runs in this order:

1. **Lookup tables first** (tables that point at nothing): stores, products, customers.
2. **Child tables next** (tables that point at a lookup table): store locations.
3. **The main fact table last** (it points at everything): orders.

Here is the practice database. Compare it to your `EmployeeDB`:

| Practice table | Kind | Like this `EmployeeDB` table |
|---|---|---|
| `stores (storeid serial, storename UNIQUE)` | lookup with a generated id | `departments`, `jobtitles`, `divisions` |
| `products (productid serial, productname UNIQUE, category)` | lookup with an extra column | `departments` (it has a description) |
| `customers (loyaltyid PRIMARY KEY, ...)` | key comes **from the CSV** | `personaldetails` (key is the SSN) |
| `storelocations (locationid serial, storeid FK, ...)` | child of a lookup, no UNIQUE | `divisionlocations`, `jobdescriptions` |
| `orders (orderid serial, ordercode, FKs, price CHECK > 0)` | fact table | `employeedetails` |

## A2. Create the practice tables

In Assignment 4 you built tables with the ERD tool. Here the guide simply runs the SQL for you. `DROP TABLE IF EXISTS` lets you re-run this cell any time to start the practice over from empty tables.

Notice the three kinds of constraint. Each one changes how the loader has to behave:
- `UNIQUE` on `storename` and `productname`: the database refuses a second "Downtown". We will use that to skip duplicates.
- `PRIMARY KEY` on `loyaltyid`: same idea, the key comes from the CSV.
- `CHECK (price > 0)`: a missing or zero price will be **rejected by the database**. We must deal with those rows before inserting.

In [4]:
PRACTICE_SCHEMA_SQL = '''
CREATE SCHEMA IF NOT EXISTS practice;
DROP TABLE IF EXISTS practice.orders, practice.storelocations, practice.customers,
                     practice.stores, practice.products;

CREATE TABLE practice.customers (
    loyaltyid   varchar(10)  PRIMARY KEY,
    name        varchar(100) NOT NULL,
    email       varchar(100),
    signupdate  date
);
CREATE TABLE practice.stores (
    storeid     serial PRIMARY KEY,
    storename   varchar(50) NOT NULL UNIQUE
);
CREATE TABLE practice.storelocations (
    locationid  serial PRIMARY KEY,
    storeid     integer REFERENCES practice.stores (storeid),
    street      varchar(100),
    city        varchar(50),
    state       varchar(2)
);
CREATE TABLE practice.products (
    productid   serial PRIMARY KEY,
    productname varchar(50) NOT NULL UNIQUE,
    category    varchar(30)
);
CREATE TABLE practice.orders (
    orderid     serial PRIMARY KEY,
    ordercode   integer,
    loyaltyid   varchar(10) REFERENCES practice.customers (loyaltyid),
    storeid     integer REFERENCES practice.stores (storeid),
    productid   integer REFERENCES practice.products (productid),
    orderdate   date,
    price       numeric(8, 2) NOT NULL CHECK (price > 0)
);
'''

conn = psycopg2.connect(**DB_PARAMS)
with conn:                       # commits if the block succeeds, rolls back if it fails
    with conn.cursor() as cur:
        cur.execute(PRACTICE_SCHEMA_SQL)
conn.close()
print("Practice tables created (empty).")

Practice tables created (empty).


## A3. Read the CSV as text, then look at it

Always read with `dtype=str`. If pandas guesses types, a ZIP code like `07102` silently becomes the number `7102`, and an id can turn into `436822.0`. Reading everything as text first means **you** decide every conversion in the next step.

Then look before you load: how many rows, and which columns have blanks?

In [5]:
raw = pd.read_csv("practice_orders.csv", dtype=str)
print(raw.shape)
raw.head(3)

(400, 13)


,OrderCode,LoyaltyID,CustomerName,CustomerEmail,SignupDate,Store,StoreStreet,StoreCity,StoreState,Product,Category,Price,OrderDate
0,50001,L1046,Dev Patel,dev.patel46@example.com,11/11/2021,Downtown,101 Main St,Bryan,TX,Croissant,Bakery,3.40,01/22/2025
1,50002,L1027,Ben Smith,ben.smith27@example.com,11/03/2021,Southside,4500 Texas Ave,College Station,TX,Croissant,Bakery,3.40,04/14/2025
2,50003,L1008,Ethan Garcia,ethan.garcia8@example.com,02/08/2022,Southside,4500 Texas Ave,College Station,TX,Green Tea,Tea,3.25,09/10/2025


In [6]:
raw.isna().sum()

OrderCode         0
LoyaltyID         0
CustomerName      0
CustomerEmail     0
SignupDate       17
Store             0
StoreStreet       0
StoreCity         0
StoreState        0
Product           0
Category          0
Price             3
OrderDate         0
dtype: int64

Two problems already: **17 blank signup dates** and **3 blank prices**. A blank signup date is acceptable (the column allows NULL). A blank price is not, because `price` is `NOT NULL CHECK (price > 0)`.

## A4. Clean and validate in pandas, BEFORE touching the database

This is the most important habit in Part A. Fix types and catch bad rows **in pandas**, where you can see them. If you let Postgres find them, the first bad row stops the whole load with an error.

**Dates.** The CSV stores dates as text like `11/11/2021`. If you insert that text, Postgres has to guess whether it means November 11 or the 11th of November, and the guess depends on the server's settings (a computer set up for day-first dates will reject `11/25/2021`). Converting to a real Python `date` first removes the guessing: the driver sends it in an unambiguous format.

`format="%m/%d/%Y"` says exactly how to read it. `errors="coerce"` turns anything unreadable into a blank (`NaT`) instead of crashing, so we can count the failures.

In [7]:
df = raw.copy()

# Strip stray spaces from every text column (" Downtown" and "Downtown" would become two stores).
for col in df.columns:
    df[col] = df[col].str.strip()

df["SignupDate"] = pd.to_datetime(df["SignupDate"], format="%m/%d/%Y", errors="coerce").dt.date
df["OrderDate"]  = pd.to_datetime(df["OrderDate"],  format="%m/%d/%Y", errors="coerce").dt.date
df["Price"]      = pd.to_numeric(df["Price"], errors="coerce")
df["OrderCode"]  = pd.to_numeric(df["OrderCode"], errors="coerce").astype("Int64")

print("Signup dates that are blank or unreadable:", df["SignupDate"].isna().sum())
print("Order dates that are blank or unreadable: ", df["OrderDate"].isna().sum())

Signup dates that are blank or unreadable: 17
Order dates that are blank or unreadable:  0


**Checkpoint:** the 17 blank signup dates were already blank in the CSV, so no date *failed* to convert. If this number were larger than the blanks you counted in A3, some dates had a format you did not expect, and you would look at those rows before going on.

**Bad rows.** Never invent a value to get past a constraint. Writing `0` for a missing price would be rejected by the CHECK anyway, and even if it were not, every average you compute later would be wrong. The professional move: set bad rows aside in a **rejects** file, load the good rows, and report how many you rejected and why.

In [8]:
bad_price = df["Price"].isna() | (df["Price"] <= 0)

rejects = df[bad_price]
good = df[~bad_price].copy()

rejects.to_csv("practice_rejects.csv", index=False)
print(f"Rows read: {len(df)}   good: {len(good)}   rejected: {len(rejects)}")
rejects[["OrderCode", "Product", "Price"]]

Rows read: 400   good: 396   rejected: 4


,OrderCode,Product,Price
17,50018,Blueberry Muffin,NaN
203,50204,Green Tea,NaN
250,50251,Croissant,0.0
388,50389,Blueberry Muffin,NaN


Three blanks plus one price of `0`: four rejects, 396 good rows.

One last helper. pandas uses its own "missing" markers (`NaN`, `NaT`, `<NA>`) and its own number types. The database driver understands Python's `None`, `int` and `float`. `clean()` converts one value so it can be inserted.

In [9]:
def clean(value):
    # Turn pandas blanks into None (SQL NULL) and numpy numbers into plain Python numbers.
    if pd.isna(value):
        return None
    if hasattr(value, "item"):
        return value.item()
    return value

print(clean(float("nan")), clean(good["OrderCode"].iloc[0]), clean(good["SignupDate"].iloc[0]))

None 50001 2021-11-11


## A5. Four loading patterns

Every table in a normalized database is loaded with one of four patterns. Learn to recognize which one a table needs.

All four share one goal: **the loader must be safe to run twice.** In real pipelines loaders get re-run all the time (a crash halfway, new data arriving, a teammate running it again). A loader that doubles the data on a second run is a broken loader. Each pattern below has its own way of saying "skip it if it is already there".

### Pattern 1: lookup table with a UNIQUE name (`stores`, `products`)

Insert each **distinct** value once. `ON CONFLICT (storename) DO NOTHING` tells Postgres: if this name already exists (the UNIQUE constraint would be violated), quietly skip it. That makes it safe to re-run.

The `%s` placeholders are filled in by the driver, never by string formatting. That keeps names with apostrophes (`Joe's`) from breaking the SQL.

### Pattern 1b: read the generated ids back

`storeid` is `serial`, so Postgres picks the numbers. The fact table needs them, so after loading we read them back into a Python dictionary: `{"Downtown": 1, "Northgate": 2, ...}`.

### Pattern 2: table whose key comes from the CSV (`customers`)

The same customer appears on many orders, so first `drop_duplicates` on the key. Then `ON CONFLICT (loyaltyid) DO NOTHING` works because a primary key is unique too.

### Pattern 3: child table with NO unique constraint (`storelocations`)

`storelocations` has no UNIQUE column, so `ON CONFLICT` has nothing to detect. Instead write the check into the INSERT itself: `INSERT ... SELECT ... WHERE NOT EXISTS (...)` inserts the row **only if** no location exists yet for that store.

### Pattern 4: the fact table (`orders`)

One row per CSV row. Each foreign key comes from the dictionaries in Pattern 1b. The same `WHERE NOT EXISTS` trick uses the order's own identifier (`ordercode`) to skip orders that were loaded on an earlier run.

### All or nothing

Everything happens inside one `with conn:` block. If anything fails halfway, Postgres **rolls back** every insert from this run, so you never end up with half a load. Fix the problem and run again.

In [10]:
def load_practice(good):
    conn = psycopg2.connect(**DB_PARAMS)
    try:
        with conn:                                   # one transaction: all or nothing
            with conn.cursor() as cur:

                # ---- Pattern 1: lookup tables ----
                for name in good["Store"].unique():
                    cur.execute(
                        "INSERT INTO practice.stores (storename) VALUES (%s) "
                        "ON CONFLICT (storename) DO NOTHING;",
                        (name,))

                products = good[["Product", "Category"]].drop_duplicates("Product")
                for _, r in products.iterrows():
                    cur.execute(
                        "INSERT INTO practice.products (productname, category) VALUES (%s, %s) "
                        "ON CONFLICT (productname) DO NOTHING;",
                        (r["Product"], r["Category"]))

                # ---- Pattern 1b: read the generated ids back into dictionaries ----
                cur.execute("SELECT storename, storeid FROM practice.stores;")
                store_ids = dict(cur.fetchall())
                cur.execute("SELECT productname, productid FROM practice.products;")
                product_ids = dict(cur.fetchall())

                # ---- Pattern 2: key comes from the CSV ----
                customers = good[["LoyaltyID", "CustomerName", "CustomerEmail", "SignupDate"]] \
                                .drop_duplicates("LoyaltyID")
                for _, r in customers.iterrows():
                    cur.execute(
                        "INSERT INTO practice.customers (loyaltyid, name, email, signupdate) "
                        "VALUES (%s, %s, %s, %s) ON CONFLICT (loyaltyid) DO NOTHING;",
                        (r["LoyaltyID"], r["CustomerName"], clean(r["CustomerEmail"]),
                         clean(r["SignupDate"])))

                # ---- Pattern 3: child table, no UNIQUE, so check with NOT EXISTS ----
                locations = good[["Store", "StoreStreet", "StoreCity", "StoreState"]] \
                                .drop_duplicates("Store")
                for _, r in locations.iterrows():
                    sid = store_ids[r["Store"]]
                    cur.execute(
                        '''INSERT INTO practice.storelocations (storeid, street, city, state)
                           SELECT %s, %s, %s, %s
                           WHERE NOT EXISTS (SELECT 1 FROM practice.storelocations
                                             WHERE storeid = %s);''',
                        (sid, r["StoreStreet"], r["StoreCity"], r["StoreState"], sid))

                # ---- Pattern 4: fact table, foreign keys from the dictionaries ----
                for _, r in good.iterrows():
                    cur.execute(
                        '''INSERT INTO practice.orders
                               (ordercode, loyaltyid, storeid, productid, orderdate, price)
                           SELECT %s, %s, %s, %s, %s, %s
                           WHERE NOT EXISTS (SELECT 1 FROM practice.orders
                                             WHERE ordercode = %s);''',
                        (clean(r["OrderCode"]), r["LoyaltyID"],
                         store_ids[r["Store"]], product_ids[r["Product"]],
                         clean(r["OrderDate"]), clean(r["Price"]),
                         clean(r["OrderCode"])))

        print("Load finished and committed.")
    finally:
        conn.close()

## A6. Run it, run it again, and prove nothing doubled

A loader is not finished when it runs without errors. It is finished when you can **prove** the database holds what the CSV said it should. That proof is called a **reconciliation**: for each table, what count do you *expect* from the CSV, and what count is *actually* in the database?

In [11]:
PRACTICE_TABLES = ["stores", "products", "customers", "storelocations", "orders"]

def table_counts(schema, tables):
    with engine.connect() as conn:
        return {t: conn.execute(text(f"SELECT COUNT(*) FROM {schema}.{t};")).scalar() for t in tables}

load_practice(good)
after_run_1 = table_counts("practice", PRACTICE_TABLES)

load_practice(good)                      # second run: should add NOTHING
after_run_2 = table_counts("practice", PRACTICE_TABLES)

expected = {
    "stores":         good["Store"].nunique(),
    "products":       good["Product"].nunique(),
    "customers":      good["LoyaltyID"].nunique(),
    "storelocations": good["Store"].nunique(),     # one location per store
    "orders":         len(good),                   # one order per good CSV row
}

recon = pd.DataFrame({"expected_from_csv": expected,
                      "after_run_1": after_run_1,
                      "after_run_2": after_run_2})
recon["ok"] = (recon["expected_from_csv"] == recon["after_run_1"]) & \
              (recon["after_run_1"] == recon["after_run_2"])
recon

Load finished and committed.


Load finished and committed.


,expected_from_csv,after_run_1,after_run_2,ok
stores,4,4,4,True
products,8,8,8,True
customers,60,60,60,True
storelocations,4,4,4,True
orders,396,396,396,True


Every row says `ok = True`: the counts match the CSV, and the second run added nothing.

**Try it (2 minutes, worth it):** in Pattern 4, delete the `WHERE NOT EXISTS (...)` part and the last `clean(r["OrderCode"])`, then re-run A2, the loader cell and this cell. Watch `orders` double on the second run. Put it back afterwards. That is the bug this pattern prevents.

### Now you: the EmployeeDB loader

In your own Assignment 5 notebook you write `load_employees()` for the 7 `EmployeeDB` tables. Before you write code, decide for each table which pattern it needs, and what its expected count is. The instructions PDF lists the CSV column that feeds each table column.

Hints, not answers:
- Which three tables are pure lookups with a UNIQUE name?
- `personaldetails` gets its key from the CSV. Which pattern is that?
- `divisionlocations` and `jobdescriptions` have no UNIQUE constraint. One of them is keyed by **two** ids. How do you write the `NOT EXISTS` check for two columns?
- `employeedetails` has no UNIQUE on `ssn` or `employeeid`. What will you use in its `NOT EXISTS` check, and why is that column safe to use?

# Part B: SQL, worked examples

In Part B of the assignment you answer business questions with **your own SQL**. Here is every technique you will need, shown on the practice tables.

A small helper runs a query and returns a DataFrame. `text()` is required by SQLAlchemy 2.x for raw SQL. Values go in through `params`, never by pasting them into the string.

In [12]:
def q(sql, params=None):
    return pd.read_sql(text(sql), engine, params=params)

### B1. JOIN + GROUP BY: one row per group

"How many orders and how much revenue did each store bring in?" The store **name** lives in `stores`, the money lives in `orders`, so JOIN them on the key, then GROUP BY the name.

Rule of thumb: every column in SELECT is either in the GROUP BY or inside an aggregate (`COUNT`, `SUM`, `AVG`, `MIN`, `MAX`).

In [13]:
q('''
SELECT s.storename, COUNT(*) AS orders, SUM(o.price) AS revenue
FROM practice.orders o
JOIN practice.stores s ON o.storeid = s.storeid
GROUP BY s.storename
ORDER BY revenue DESC;
''')

,storename,orders,revenue
0,Downtown,114,642.40
1,Northgate,96,553.05
2,Airport,93,536.70
3,Southside,93,466.00


### B2. Filter with a parameter

`:cat` is a placeholder. The value travels separately in the `params` dictionary. Filter on the **name** a person would use (`'Tea'`), not on an id number you looked up by hand: ids are generated and can differ on someone else's machine.

In [14]:
q('''
SELECT p.productname, COUNT(*) AS orders
FROM practice.orders o
JOIN practice.products p ON o.productid = p.productid
WHERE p.category = :cat
GROUP BY p.productname
ORDER BY orders DESC;
''', {"cat": "Tea"})

,productname,orders
0,Green Tea,53
1,Chai Tea,47


### B3. Compare against an average: subquery + HAVING

`WHERE` filters rows **before** grouping. `HAVING` filters groups **after** grouping. The subquery in parentheses computes one number (the overall average) that every group is compared to.

In [15]:
q('''
SELECT p.productname, ROUND(AVG(o.price), 2) AS avg_order_value
FROM practice.orders o
JOIN practice.products p ON o.productid = p.productid
GROUP BY p.productname
HAVING AVG(o.price) > (SELECT AVG(price) FROM practice.orders)
ORDER BY avg_order_value DESC;
''')

,productname,avg_order_value
0,Latte,8.04
1,Cold Brew,7.74
2,Chai Tea,6.39


### B4. Dates and buckets: AGE, EXTRACT and CASE

This only works because `signupdate` is a real `date` (Part A). `AGE(CURRENT_DATE, d)` gives an interval like "3 years 2 mons"; `EXTRACT(YEAR FROM ...)` pulls out the whole years. `CASE` turns a number into a label so you can group by it. Blank dates are excluded on purpose, and you should say so when you report the result.

In [16]:
q('''
SELECT CASE WHEN EXTRACT(YEAR FROM AGE(CURRENT_DATE, signupdate)) < 3
            THEN 'Under 3 years' ELSE '3 years or more' END AS membership,
       COUNT(*) AS customers
FROM practice.customers
WHERE signupdate IS NOT NULL
GROUP BY membership
ORDER BY membership;
''')

,membership,customers
0,3 years or more,44
1,Under 3 years,14


### B5. Best per group: window function

"What is the best-selling product **at each store**?" `GROUP BY` alone gives counts for every store and product pair. `ROW_NUMBER() OVER (PARTITION BY store ORDER BY count DESC)` numbers the products 1, 2, 3 *within each store*, and the outer query keeps number 1.

In [17]:
q('''
SELECT storename, productname, orders FROM (
    SELECT s.storename, p.productname, COUNT(*) AS orders,
           ROW_NUMBER() OVER (PARTITION BY s.storename
                              ORDER BY COUNT(*) DESC, p.productname) AS rnk
    FROM practice.orders o
    JOIN practice.stores   s ON o.storeid   = s.storeid
    JOIN practice.products p ON o.productid = p.productid
    GROUP BY s.storename, p.productname
) ranked
WHERE rnk = 1
ORDER BY storename;
''')

,storename,productname,orders
0,Airport,Blueberry Muffin,17
1,Downtown,Croissant,19
2,Northgate,Cold Brew,14
3,Southside,Blueberry Muffin,17


### B6. Find what is missing: LEFT JOIN ... IS NULL

A normal JOIN drops customers who never ordered. A LEFT JOIN keeps every customer and fills the order columns with NULL when there is no match, so `WHERE o.orderid IS NULL` finds exactly the customers with no orders.

An **empty result is a valid answer**. Here it means every customer has at least one order. When your query returns nothing, check that this is true rather than assuming the query is wrong.

In [18]:
q('''
SELECT c.loyaltyid, c.name
FROM practice.customers c
LEFT JOIN practice.orders o ON o.loyaltyid = c.loyaltyid
WHERE o.orderid IS NULL;
''')

,loyaltyid,name


### B7. Data-quality check: duplicates

Constraints are not the only check. Here: does any email belong to more than one customer? `GROUP BY` the value, `HAVING COUNT(*) > 1`.

In [19]:
q('''
SELECT email, COUNT(*) AS customers_with_it
FROM practice.customers
WHERE email IS NOT NULL
GROUP BY email
HAVING COUNT(*) > 1;
''')

,email,customers_with_it


# Part C: Serving data with Streamlit

Streamlit turns a plain Python file into a web page. You write ordinary Python from top to bottom; each `st.something(...)` line puts one thing on the page (a title, a filter, a chart) in the order the lines run. No HTML or JavaScript is needed.

This part has four steps: **C1** how Streamlit runs, **C2** what each function does, **C3** a walk through `practice_app.py` block by block, **C4** run it and try it.

## C1. The one idea that explains Streamlit

**Every time someone touches a widget, Streamlit runs your whole file again, from the first line to the last.**

Pick a store in the sidebar and the file re-runs; this time the variable holding the chosen stores has the new value, so every query and chart below it uses that value. There are no callbacks or event handlers to write. That one rule explains why the code in C3 caches the connection and the query results: without caching, every click would reconnect to Postgres and re-ask every question.

A dashboard is a `.py` file, not a notebook cell. It runs in a terminal and shows up in your browser.

## C2. Streamlit vocabulary

Short definitions of every function used in `practice_app.py` and in your own `app.py`. Read the table once now, then come back to it while reading C3. The full reference is at https://docs.streamlit.io.

**Page and text**

| Function | What it does |
|---|---|
| `st.set_page_config(page_title=..., layout="wide")` | Sets the browser tab title and lets the page use the full screen width. Must be the first `st.` call in the file. |
| `st.title("...")` | Big heading at the top of the page. |
| `st.subheader("...")` | Smaller heading, used above each chart. |
| `st.warning("...")`, `st.info("...")` | A yellow or blue message box, for example "No data matches these filters". |
| `st.stop()` | Stops running the file right here, for this run. Used when there is nothing sensible to show (no store selected). |

**Filters (widgets).** Each one draws a control and **returns the user's current choice** as a normal Python value.

| Function | Returns |
|---|---|
| `st.multiselect("Label", options, default=...)` | A **list** of the chosen options. |
| `st.selectbox("Label", options)` | **One** chosen option. |
| `st.slider("Label", min, max, (low, high))` | With a tuple as the starting value: a **(low, high)** pair, a range. (Not in the practice app; your dashboard needs it.) |
| `st.sidebar.<widget>(...)` | The same widget, but placed in the left sidebar instead of the main page. `st.sidebar.header("...")` puts a heading there. |

**Layout**

| Function | What it does |
|---|---|
| `st.columns(n)` | Splits the page into `n` side-by-side columns and returns them. Use `with col:` to put things inside one, or call `col.metric(...)` on it directly. |
| `st.expander("Label")` | A collapsible box. Whatever you put inside `with st.expander(...):` stays hidden until the user clicks it. |

**Showing data**

| Function | What it does |
|---|---|
| `st.metric("Label", value)` | One big number with a label: a KPI tile. |
| `st.plotly_chart(fig, width="stretch")` | Draws a Plotly figure, stretched to the width of its column. Use `st.pyplot(fig)` for a matplotlib figure instead. |
| `st.dataframe(df, hide_index=True)` | An interactive table of a DataFrame (sortable, scrollable). |

**Caching (decorators).** You put these on the line above a function definition.

| Decorator | Use it for |
|---|---|
| `@st.cache_resource` | **Things** that should be created once and shared by every re-run: the database engine. |
| `@st.cache_data(ttl=600)` | **Results** of a function, such as a DataFrame from a query. Same inputs give the stored answer instead of re-running. `ttl=600` throws the stored answer away after 600 seconds, so new data shows up eventually. |

**The non-Streamlit pieces in the file**

| Function | What it does |
|---|---|
| `create_engine(url)` (SQLAlchemy) | Builds an engine: an object that knows how to open connections to your database. |
| `text(sql)` (SQLAlchemy) | Wraps a SQL string so `:name` placeholders can be filled from a `params` dictionary. |
| `pd.read_sql(text(sql), conn, params=...)` | Runs the query and returns the result as a DataFrame. |
| `px.bar`, `px.line`, `px.box` (Plotly Express) | Build a bar, line or box chart from a DataFrame: you name the columns for `x`, `y` and optionally `color`. `labels={...}` renames axis titles. |
| `fig.update_layout(...)` | Changes chart settings after it is built, for example the order of the bars. |

## C3. Walking through `practice_app.py`

Open `practice_app.py` next to this notebook. Below is the same file in blocks, in the order it runs, each with what it does and why. The numbers match the numbered comments in the file.

### CONFIG and page setup

```python
import pandas as pd
import plotly.express as px
import streamlit as st
from sqlalchemy import create_engine, text

DB_NAME = "EmployeeDB"
DB_USER = "postgres"
DB_PASSWORD = "admin"      # change to YOUR PostgreSQL password
DB_HOST = "localhost"
DB_PORT = "5432"

st.set_page_config(page_title="Bean Counter Coffee", layout="wide")
```

The same CONFIG idea as the notebooks: connection settings in one place at the top. `set_page_config` comes before any other `st.` call.

### 1. One engine for the whole app

```python
@st.cache_resource
def get_engine():
    return create_engine(
        f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
    )
```

Because the file re-runs on every click (C1), a plain `create_engine(...)` line would build a new engine every time. `@st.cache_resource` builds it on the first run and hands back the same one afterwards.

### 2. One helper that runs every query

```python
@st.cache_data(ttl=600)
def run_query(sql, params=None):
    with get_engine().connect() as conn:
        return pd.read_sql(text(sql), conn, params=params)
```

Every query in the app goes through this function, so caching is written once. `with ... as conn:` borrows a connection and gives it back automatically when the block ends. That is why there is **no `conn.close()` anywhere** in the app: closing it yourself in the middle of the file breaks every chart after that line.

### 3. Filters in the sidebar

```python
st.title("Bean Counter Coffee: orders dashboard")

all_stores = run_query("SELECT storename FROM practice.stores ORDER BY storename;")["storename"].tolist()
all_categories = run_query("SELECT DISTINCT category FROM practice.products ORDER BY category;")["category"].tolist()

st.sidebar.header("Filters")
chosen_stores = st.sidebar.multiselect("Stores", all_stores, default=all_stores)
chosen_category = st.sidebar.selectbox("Product category", ["All"] + all_categories)

if not chosen_stores:
    st.warning("Pick at least one store in the sidebar.")
    st.stop()
```

The options come **from the database**, not typed by hand, so a new store appears in the filter automatically. `chosen_stores` is a list (multiselect), `chosen_category` is one string (selectbox). If the user removes every store, there is nothing to show, so the app says so and stops.

### The shared filter: written once, used by every query

```python
FILTER_SQL = """
    FROM practice.orders o
    JOIN practice.stores   s ON o.storeid   = s.storeid
    JOIN practice.products p ON o.productid = p.productid
    WHERE s.storename = ANY(:stores)
      AND (:cat = 'All' OR p.category = :cat)
"""
params = {"stores": chosen_stores, "cat": chosen_category}
```

This is the most important block. Every query below is `"SELECT ..." + FILTER_SQL + "GROUP BY ..."`, so every chart obeys the same filters and they always agree.
- `:stores` and `:cat` are placeholders; the values arrive separately in `params`. Never paste a chosen value into the SQL with an f-string: a name like `O'Brien` breaks the query, and a crafted value can change what the query does (SQL injection).
- `= ANY(:stores)` accepts a whole Python list.
- `(:cat = 'All' OR p.category = :cat)` lets one query serve both cases: when "All" is chosen, the first half is true and the category condition is switched off.

### 4. KPI row

```python
kpi = run_query(
    "SELECT COUNT(*) AS orders, COALESCE(SUM(o.price), 0) AS revenue, "
    "COUNT(DISTINCT o.loyaltyid) AS customers " + FILTER_SQL,
    params,
)
c1, c2, c3 = st.columns(3)
c1.metric("Orders", f"{kpi.loc[0, 'orders']:,}")
c2.metric("Revenue", f"${kpi.loc[0, 'revenue']:,.2f}")
c3.metric("Customers", f"{kpi.loc[0, 'customers']:,}")
```

One query returns one row with three numbers. `st.columns(3)` makes three tiles side by side. `COALESCE(..., 0)` turns "no rows, so SUM is NULL" into 0. `:,` in the f-string adds thousands separators.

### 5. Bar chart: SQL groups, Plotly draws

```python
left, right = st.columns(2)

by_product = run_query(
    "SELECT p.productname, SUM(o.price) AS revenue " + FILTER_SQL +
    " GROUP BY p.productname ORDER BY revenue DESC;",
    params,
)
with left:
    st.subheader("Revenue by product")
    if by_product.empty:
        st.info("No orders match these filters.")
    else:
        fig = px.bar(by_product, x="revenue", y="productname", orientation="h",
                     labels={"revenue": "Revenue ($)", "productname": ""})
        fig.update_layout(yaxis={"categoryorder": "total ascending"})
        st.plotly_chart(fig, width="stretch")
```

The database does the grouping and sends 8 rows, one per product, instead of all 400 orders. `orientation="h"` makes horizontal bars; `categoryorder` sorts them so the biggest is on top. Always handle the empty case: some filter combinations match nothing.

### 6. Line chart over time

```python
by_month = run_query(
    "SELECT DATE_TRUNC('month', o.orderdate)::date AS month, s.storename, COUNT(*) AS orders " + FILTER_SQL +
    " GROUP BY month, s.storename ORDER BY month;",
    params,
)
with right:
    ...
        fig = px.line(by_month, x="month", y="orders", color="storename", markers=True, ...)
```

`DATE_TRUNC('month', ...)` rounds every date down to the first of its month so orders can be counted per month. `color="storename"` draws one line per store. This works only because `orderdate` was loaded as a real date in Part A.

### 7. Box plot: the one chart that needs raw rows

```python
prices = run_query("SELECT s.storename, o.price " + FILTER_SQL + ";", params)
...
    fig = px.box(prices, x="storename", y="price", points="outliers", ...)
```

A box plot shows the spread of values (median, quartiles, outliers), so Plotly needs every individual price, not a pre-computed average. `points="outliers"` draws only the unusual points as dots.

### 8. Detail table in an expander

```python
with st.expander("Show matching orders"):
    detail = run_query(
        "SELECT o.ordercode, o.orderdate, s.storename, p.productname, o.price " + FILTER_SQL +
        " ORDER BY o.orderdate DESC;",
        params,
    )
    st.dataframe(detail, width="stretch", hide_index=True)
```

Long tables make a dashboard hard to scan, so this one stays folded until someone opens it.

## C4. Run it and try it

In a terminal (not in Jupyter), with your course environment active, from the folder that holds the file:

```
streamlit run practice_app.py
```

Change the password in the file's CONFIG block first. A browser tab opens at `http://localhost:8501`. Stop the app with `Ctrl+C` in the terminal. When you save a change to the file, Streamlit shows a **Rerun** button in the top-right corner of the page.

Try each of these, and predict what will happen **before** you click:
- Remove a store in the sidebar. Every number and every chart should change. If one did not, its query would not be using `params`.
- Pick the category "Tea". The KPI tiles should drop.
- Remove every store. You should see the warning and nothing else (that is `st.stop()`).
- Add a fourth KPI tile, the average order value (`AVG(o.price)`), by changing `st.columns(3)` to 4 and adding a `c4.metric(...)` line. Save and click Rerun.

When all of that makes sense, you are ready to write `app.py` for `EmployeeDB` from the specification in the instructions.